### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import tomllib
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
from scipy.special import expit

from prettytable import PrettyTable

from src.stat_utils import *
from src.sim_utils import generate_stimulus_conditions
from src.anl_utils import load_data, get_session_data

### Load and prepare data

In [3]:
sim_results_folder = '../results/simulation'
data_folder = '../data'

sync_at_file = os.path.join(sim_results_folder, 'highres_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

with open('../config/analysis/experiment_actual.toml', 'rb') as f:
    experiment_params = tomllib.load(f)

grid_coarseness, contrast_heterogeneity = generate_stimulus_conditions(experiment_params)

In [ ]:
sync_results = np.load(sync_at_file)
num_sessions = sync_results.shape[0]
num_simulations = sync_results.shape[1]
sync_results = sync_results[:, :,::6, ::6].reshape(num_sessions, num_simulations, -1)
data_synthetic = {'Synchrony': [], 'SessionID': [], 'ContrastHeterogeneity': [], 'GridCoarseness': []}

for session_id in range(num_sessions):
    for simulation_id in range(num_simulations):
        data_synthetic['Synchrony'].extend(sync_results[session_id, simulation_id])
        data_synthetic['SessionID'].extend([session_id + 1] * 25)
        data_synthetic['ContrastHeterogeneity'].extend(contrast_heterogeneity)
        data_synthetic['GridCoarseness'].extend(grid_coarseness)

data_synthetic = pd.DataFrame(data_synthetic)
data_synthetic = data_synthetic[data_synthetic['SessionID'] >2].copy()
data_synthetic = zscore_data(data_synthetic, ['ContrastHeterogeneity', 'GridCoarseness','Synchrony', 'SessionID'])

sync_results = sync_results.mean(axis=1)

data = load_data(emp_at_file)
data_training = data[(data['SessionID']<9)].copy()

data_transfer = zscore_data(data, ['ContrastHeterogeneity', 'GridCoarseness'])

unique_sessions = data_training['SessionID'].unique()

for i, session in enumerate(unique_sessions):
    session_data = data_training[data_training['SessionID'] == session].copy()
    # Map synchrony values to each condition in each session in DataFrame
    session_data['Synchrony'] = session_data['Condition'].apply(lambda x: sync_results[session-1, x-1])
    data_training.loc[data_training['SessionID'] == session, 'Synchrony'] = session_data['Synchrony']

data_training = zscore_data(data_training, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony', 'SessionID'])

### Establish learning effect

In [ ]:
m_learning = bmb.Model(
    "Correct ~ 1 + SessionID + ContrastHeterogeneity * GridCoarseness + (1|SubjectID)",
    data=data_training,
    family="bernoulli"
)
idata_learning = m_learning.fit(draws=2000, tune=2000, target_accept=0.95)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, SessionID, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset]


Output()

In [ ]:
beta_session = idata_learning.posterior["SessionID"].values.flatten()
prob_positive = np.mean(beta_session > 0)

or_session_mean = np.exp(beta_session).mean()
or_session_low = np.percentile(np.exp(beta_session), 2.5)
or_session_high = np.percentile(np.exp(beta_session), 97.5)

print(f"Probability that the session effect is positive: {prob_positive:.2f}")
az.summary(idata_learning, var_names=["Intercept", "SessionID", "ContrastHeterogeneity", "GridCoarseness"], hdi_prob=0.95)

In [ ]:
beta0 = idata_learning.posterior["Intercept"].values.flatten()

# Predicted accuracy at -1 SD and +1 SD synchrony
p_low = expit(beta0 - beta_session)
p_high = expit(beta0 + beta_session)


table = PrettyTable()
table.field_names = ["Metric", "Value"]
table.add_row(["Probability low session", p_low.mean()])
table.add_row(["Probability high session", p_high.mean()])
table.add_row(["Difference", (p_high - p_low).mean()])

print(table)

### Validate assumption of local learning

In [ ]:

## C(SessionID, Treatment(reference=9))

m_s0 = bmb.Model(
    "Correct ~ 1 + C(SessionID) + ContrastHeterogeneity * GridCoarseness + (1|SubjectID)",
    data=data_transfer,
    family="bernoulli"
)
id_s0 = m_s0.fit(draws=2000, tune=2000, target_accept=0.95,
                 idata_kwargs={"log_likelihood": True})


In [ ]:
beta_transfer = id_s0.posterior["C(SessionID)"].values.flatten()

probability = np.mean(beta_transfer <= 0)
hdi = az.hdi(beta_transfer, hdi_prob=0.95)

# Convert to Odds Ratios
or_hdi = np.exp(hdi)
or_mean = np.exp(beta_transfer).mean()

print(f"P(S9 < S2) = {probability:.3f}")
print(f"95% CrI for log-odds difference = {hdi}")
print(f"Mean OR = {or_mean:.3f}")
print(f"95% CrI for OR = {or_hdi}")


### Test learning mechanism

**Logic**

1. Learning increasing coupling strength (part of V1 model)
2. Increased coupling strength widens the synchrony range - Arnold tongue (first test using model simulations only)
3. Widened synchrony widens performance range - behavioral Arnold tongue (second test using empirical data)


In [ ]:
m_synthetic = bmb.Model(
    formula='Synchrony ~ 1 + SessionID * (ContrastHeterogeneity + GridCoarseness)',
    data=data_synthetic,
    family='gaussian'
)

idata_synthetic = m_synthetic.fit(draws=2000, tune=2000, target_accept=0.95)


In [ ]:
beta_session_ch = idata_synthetic.posterior['SessionID:ContrastHeterogeneity'].values.flatten()
beta_session_gc = idata_synthetic.posterior['SessionID:GridCoarseness'].values.flatten()

prob_positive_sess_ch = (beta_session_ch < 0).mean()
prob_positive_sess_gc = (beta_session_gc > 0).mean()

print(f"Probability that the session effect on ContrastHeterogeneity is positive: {prob_positive_sess_ch:.2f}")
print(f"Probability that the session effect on GridCoarseness is positive: {prob_positive_sess_gc:.2f}")

az.summary(idata_synthetic, var_names=["Intercept", "SessionID", "ContrastHeterogeneity", "GridCoarseness", "SessionID:ContrastHeterogeneity", "SessionID:GridCoarseness"], hdi_prob=0.95)